# Regularization for Deep Learning — a hands-on MNIST notebook

*Companion to Chapter 7 of Goodfellow, Bengio & Courville — "Regularization for Deep Learning" (§7.1–7.3).*

This notebook turns every idea from the interactive lesson into **runnable MNIST experiments**. Each section maps 1:1 to a "station" of the lesson:

| # | Station | What we run on MNIST |
|---|---------|----------------------|
| 0 | The core problem | Make a model overfit and *measure* the train–validation gap |
| 1 | What regularization is | The two-force "tug of war" in weight space (the FIG 01 picture, live) |
| 2 | L² weight decay | `weight_decay`, the `(1 − εα)` per-step shrink, AdamW |
| 3 | L² through the Hessian | Eigenvalues of XᵀX and the `λ/(λ+α)` rescaling |
| 4 | L¹ & sparsity | Soft-thresholding a linear classifier → **exact zeros**, pixel selection |
| 5 | L¹ vs L² & priors | Gaussian vs Laplace, weight histograms (spike at 0) |
| 6 | Penalties as constraints | α ↔ region size *k* |
| 7 | Explicit constraints | Per-column **max-norm** reprojection |
| 8 | Under-constrained problems | Singular XᵀX + αI, separable-data runaway, the pseudoinverse |

> **The one idea to carry off the landscape:** a norm penalty is a *second force* on the weights. The data term pulls toward the fit `w*`; the penalty pulls toward `0`; the solution `w̃` is the balance. Everything else is just *which shape of pull* and *how hard*.

**Runtime:** works on CPU; a GPU (Runtime → Change runtime type → GPU) makes it snappier. Run the cells top to bottom.


## Setup — imports, data, and a reusable training loop

We load MNIST once and build every experiment on top of it. Read the comments in each cell — they explain *why*, not just *what*.


In [ ]:
# --- imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Standard MNIST normalization (mean/std of the training set).
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_full = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_full  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
print('Full train set:', len(train_full), '| Test set:', len(test_full))

### Why a *small* training subset?

Chapter 7 exists because of one symptom: **99% on training, 82% on validation** — a large generalization gap. To make that gap appear reliably (and quickly), we deliberately train on a **small subset** of MNIST. A large network on ~1,000 images memorises them in a few epochs — the perfect stage for watching regularization work.

> **Rule of thumb from the lesson:** a big train–validation gap ⇒ add regularization or data. Poor scores *even on training* ⇒ the model is too small (underfitting), which is a different problem.


In [ ]:
# Deliberately tiny training set so overfitting is dramatic and fast to reproduce.
N_TRAIN = 1000
sub_idx = np.random.choice(len(train_full), N_TRAIN, replace=False)
train_small = Subset(train_full, sub_idx)

train_loader = DataLoader(train_small, batch_size=64, shuffle=True)
val_loader   = DataLoader(test_full,   batch_size=512, shuffle=False)
print(f'Training on {len(train_small)} images; validating on {len(test_full)}.')

### The model and a flexible training loop

One MLP and one `train()` function power most of the notebook. The function exposes exactly the knobs the chapter talks about:

- `weight_decay` → **L² / weight decay** (folded into the optimizer, station 2)
- `l1_lambda` → an **explicit L¹ penalty** on the weights (station 4)
- `max_norm` → **per-column reprojection / max-norm** after each step (station 7)
- `optimizer_name` → `sgd`, `adam`, or `adamw` (for the AdamW discussion in station 2)

Note we penalize **weights only, never biases** — exactly as the chapter argues (a bias controls a single variable and is fit accurately from little data, so regularizing it just adds underfitting).


In [ ]:
class MLP(nn.Module):
    """A plain 2-hidden-layer MLP. Big enough to overfit 1,000 images."""
    def __init__(self, hidden=256, p_drop=0.0):
        super().__init__()
        self.fc1  = nn.Linear(28*28, hidden)
        self.fc2  = nn.Linear(hidden, hidden)
        self.fc3  = nn.Linear(hidden, 10)
        self.drop = nn.Dropout(p_drop)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x)); x = self.drop(x)
        x = F.relu(self.fc2(x)); x = self.drop(x)
        return self.fc3(x)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb.view(xb.size(0), -1))  # flatten: no-op for MLP, needed for bare Linear
        loss_sum += F.cross_entropy(out, yb, reduction='sum').item()
        correct  += (out.argmax(1) == yb).sum().item()
        tot      += yb.size(0)
    return loss_sum / tot, correct / tot

def weight_l2_norm(model):
    """L2 norm of all *weight* tensors (biases excluded)."""
    sq = sum(p.pow(2).sum().item() for n, p in model.named_parameters() if 'weight' in n)
    return sq ** 0.5

In [ ]:
def train(model, epochs=60, lr=0.05, weight_decay=0.0, l1_lambda=0.0,
          max_norm=None, optimizer_name='sgd', verbose_every=20):
    """One training loop with all the chapter's regularizers as switches."""
    model.to(device)
    params = model.parameters()
    if optimizer_name == 'sgd':
        opt = torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'adamw':
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    hist = {k: [] for k in ['train_loss','train_acc','val_loss','val_acc','wnorm']}
    for ep in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)

            # --- explicit L1 penalty on weights only (station 4) ---
            if l1_lambda > 0:
                l1 = sum(p.abs().sum() for n, p in model.named_parameters() if 'weight' in n)
                loss = loss + l1_lambda * l1

            loss.backward()
            opt.step()

            # --- per-column max-norm reprojection (station 7) ---
            if max_norm is not None:
                with torch.no_grad():
                    for n, p in model.named_parameters():
                        if 'weight' in n:
                            # each row = incoming weights of one output unit
                            norms = p.norm(dim=1, keepdim=True)
                            scale = (max_norm / (norms + 1e-12)).clamp(max=1.0)
                            p.mul_(scale)

        tl, ta = evaluate(model, train_loader)
        vl, va = evaluate(model, val_loader)
        for k, v in zip(hist, [tl, ta, vl, va, weight_l2_norm(model)]):
            hist[k].append(v)
        if verbose_every and (ep + 1) % verbose_every == 0:
            print(f'epoch {ep+1:3d} | train_acc {ta:.3f} | val_acc {va:.3f} | |w| {hist["wnorm"][-1]:.2f}')
    return hist

## Station 0 — The core problem: overfitting

Before any regularization, let's *see* the disease. We train the MLP with **no** regularization and watch training accuracy race to ~100% while validation accuracy stalls far below. That gap is exactly what every method in this chapter exists to close.


In [ ]:
torch.manual_seed(0)
model_overfit = MLP(hidden=256, p_drop=0.0)
hist_overfit  = train(model_overfit, epochs=60, lr=0.05, weight_decay=0.0)

gap = hist_overfit['train_acc'][-1] - hist_overfit['val_acc'][-1]
print(f"\nFinal train acc {hist_overfit['train_acc'][-1]:.3f} | "
      f"val acc {hist_overfit['val_acc'][-1]:.3f} | GAP {gap:.3f}")

In [ ]:
def plot_curves(hists, labels, metric='acc', title=''):
    plt.figure(figsize=(8,5))
    for h, lab in zip(hists, labels):
        line, = plt.plot(h['val_'+metric],   '-',  label=f'{lab} · val')
        plt.plot(h['train_'+metric], '--', color=line.get_color(), alpha=0.6,
                 label=f'{lab} · train')
    plt.xlabel('epoch'); plt.ylabel(metric); plt.grid(alpha=0.3)
    plt.legend(fontsize=8); plt.title(title or 'Train (dashed) vs Validation (solid)')
    plt.show()

plot_curves([hist_overfit], ['no reg'], 'acc', 'Station 0 — overfitting: the gap opens up')

The dashed line (train) climbs toward 1.0; the solid line (val) plateaus well below. The vertical distance between them **is** the overfitting we now attack. Keep this run as the baseline to beat.


## Station 1 — What regularization *is*: a tug of war

**Definition.** Regularization is *any modification to a learning algorithm intended to reduce generalization error but not training error.* The classical form adds a **parameter norm penalty** Ω(θ) to the cost:

$$\tilde J(\theta;X,y) = J(\theta;X,y) + \alpha\,\Omega(\theta), \qquad \alpha \in [0,\infty)$$

Two forces act on the weights:

- **Data term** `J` — its gradient pulls toward `w*`, the training optimum.
- **Penalty** `αΩ` — a spring pulling toward the origin `0`.
- **α** — the dial. `α=0` recovers the unregularized problem; larger `α` pulls harder. It's a *hyperparameter tuned on validation*, not learned.
- **The solution `w̃`** sits where the two gradients cancel: `∇J(w̃) = −αw̃`.

We can draw this exactly (the lesson's FIG 01). We approximate the data loss by a quadratic bowl `J(w) = ½(w−w*)ᵀH(w−w*)` and add an L² penalty. The regularized optimum has the closed form `w̃ = (H+αI)⁻¹ H w*`.


In [ ]:
# The tug of war in 2-D weight space (reproduces FIG 01 / the contour bench).
w_star = np.array([3.0, 2.0])                  # data optimum
H = np.array([[3.8, 0.0],                      # steep along w1 ...
              [0.0, 0.7]])                      # ... flat along w2

def w_tilde_2d(alpha):
    return np.linalg.solve(H + alpha*np.eye(2), H @ w_star)

# loss contours
g = np.linspace(-0.5, 4, 240)
W1, W2 = np.meshgrid(g, g)
D1, D2 = W1 - w_star[0], W2 - w_star[1]
J = 0.5*(H[0,0]*D1**2 + H[1,1]*D2**2)

plt.figure(figsize=(6.5,6))
plt.contour(W1, W2, J, levels=18, cmap='viridis', alpha=0.6)
path = np.array([w_tilde_2d(a) for a in np.linspace(0, 12, 60)])
plt.plot(path[:,0], path[:,1], color='crimson', lw=2, label='w̃ as α grows →')
plt.scatter(*w_star, color='k', zorder=5, label='w* (data optimum)')
plt.scatter(0, 0, color='crimson', marker='x', s=80, zorder=5, label='origin (penalty pulls here)')
plt.scatter(*w_tilde_2d(0.7), color='orange', s=80, zorder=6, label='w̃ at α=0.7')
plt.xlabel('w₁ (steep direction)'); plt.ylabel('w₂ (flat direction)')
plt.legend(fontsize=8); plt.title('Station 1 — data force vs penalty force')
plt.axhline(0, color='gray', lw=.5); plt.axvline(0, color='gray', lw=.5)
plt.show()

print('Notice: as α grows, w̃ slides from w* toward 0 — and it gives up the FLAT w₂ first.')

The red curve is the *truce line*: every point is the balance `w̃` for some `α`. As `α` increases, `w̃` slides from `w*` toward the origin — and it surrenders the **flat** direction (`w₂`) long before the **steep** one (`w₁`). That preference is the whole story of station 3.

**Why weights, not biases?** A weight couples two variables, so fitting it well needs seeing both under many conditions — prone to overfit, worth regularizing. A bias controls a single variable and is fit accurately from far less data; regularizing it mostly just adds underfitting. That's why our `train()` loop penalizes `weight` tensors only.


## Station 2 — L² regularization = weight decay

The most common penalty is half the squared L² norm:

$$\Omega(\theta)=\tfrac12\lVert w\rVert_2^2 \;\Rightarrow\; \tilde J(w)=J(w)+\tfrac{\alpha}{2}w^\top w,\qquad \nabla_w\tilde J = \nabla_w J + \alpha w.$$

A single SGD step (learning rate ε) becomes:

$$w \leftarrow (1-\varepsilon\alpha)\,w \;-\; \varepsilon\,\nabla_w J.$$

Before every ordinary update the weights are multiplied by a constant **just under 1** — they *decay*. Let's read that factor off, then train MNIST across a sweep of `weight_decay` and watch the gap close and the weight norm shrink.


In [ ]:
# --- decay-per-step meter (the live gauge from the lesson) ---
print('shrink factor (1 - eps*alpha) and per-step %decay:')
for eps, alpha in [(0.05, 0.5), (0.01, 0.01), (0.10, 1e-4), (0.05, 1e-2)]:
    factor = 1 - eps*alpha
    print(f'  eps={eps:<5} alpha={alpha:<7} -> factor {factor:.5f}  '
          f'(decays {100*(1-factor):.3f}% per step)')
print('\nKey coupling: the *effective* decay depends on the learning rate eps.')
print('Change eps and you silently change how hard weight decay bites -> AdamW decouples them.')

In [ ]:
# --- sweep weight_decay on MNIST ---
hists_wd = {}
for wd in [0.0, 1e-3, 1e-2, 5e-2]:
    torch.manual_seed(0)
    m = MLP(hidden=256)
    hists_wd[wd] = train(m, epochs=60, lr=0.05, weight_decay=wd, verbose_every=0)
    h = hists_wd[wd]
    print(f'weight_decay={wd:<6} | train {h["train_acc"][-1]:.3f} | '
          f'val {h["val_acc"][-1]:.3f} | gap {h["train_acc"][-1]-h["val_acc"][-1]:.3f} | '
          f'|w| {h["wnorm"][-1]:.2f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,5))
for wd, h in hists_wd.items():
    ax[0].plot(h['val_acc'],  label=f'wd={wd}')
    ax[1].plot(h['wnorm'],    label=f'wd={wd}')
ax[0].set_title('Validation accuracy vs weight_decay'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('val acc')
ax[1].set_title('Weight norm |w| shrinks as wd grows');  ax[1].set_xlabel('epoch'); ax[1].set_ylabel('|w|')
for a in ax: a.grid(alpha=0.3); a.legend()
plt.show()
print('More weight decay -> smaller |w| -> smaller train/val gap (until too much underfits).')

### The Adam vs AdamW catch

With Adam, naïve L² and *true* weight decay differ, because Adam rescales gradients per-parameter — which drags the L² term along with them. **AdamW** ("decoupled weight decay") applies the `(1−εα)` shrink directly to the weights, sidestepping that interaction. It's now standard for Transformers. Quick side-by-side on our subset:


In [ ]:
for opt_name in ['adam', 'adamw']:
    torch.manual_seed(0)
    m = MLP(hidden=256)
    h = train(m, epochs=40, lr=1e-3, weight_decay=1e-2,
              optimizer_name=opt_name, verbose_every=0)
    print(f'{opt_name:6s} | val acc {h["val_acc"][-1]:.3f} | |w| {h["wnorm"][-1]:.2f}')
print('\nTypical recipe: AdamW with weight_decay=0.01; sweep 1e-4 -> 1e-2 on validation.')

## Station 3 — L² through the Hessian: which directions survive

Approximate the loss near its optimum by a quadratic bowl whose curvature is the **Hessian** `H`. In `H`'s eigenbasis, L² rescales each direction independently:

$$\tilde w = (H+\alpha I)^{-1}H\,w^*,\qquad \tilde w_i = \frac{\lambda_i}{\lambda_i+\alpha}\,w^*_i.$$

The factor `λ/(λ+α)` is near **1** when `λ ≫ α` (steep, high-curvature directions are *kept*) and near **0** when `λ ≪ α` (flat directions are *decayed away*). For a linear model `J(w)=½‖Xw−y‖²` the Hessian is literally `H = XᵀX`, so we can compute all of this exactly. We downsample MNIST to 7×7 so the Hessian is a tidy 49×49 matrix.


In [ ]:
def get_xy(dataset, classes, n, pool=4, seed=0):
    """Downsampled MNIST features + labels for the requested digit classes."""
    rng = np.random.default_rng(seed)
    targets = dataset.targets.numpy()
    idx = np.where(np.isin(targets, classes))[0]
    idx = rng.choice(idx, size=min(n, len(idx)), replace=False)
    X = torch.stack([dataset[i][0] for i in idx])          # [n,1,28,28]
    X = F.avg_pool2d(X, pool).reshape(len(idx), -1).numpy()  # [n, (28/pool)^2]
    y = targets[idx]
    return X, y

# Two-class linear regression (0 vs 1) -> clean, small Hessian.
X, y = get_xy(train_full, [0, 1], 800, pool=4)
y = np.where(y == 0, -1.0, 1.0)
X = (X - X.mean(0)) / (X.std(0) + 1e-8)      # standardize features
m = len(X)

H = (X.T @ X) / m                              # Hessian of the MSE = X^T X
w_star = np.linalg.solve(H + 1e-6*np.eye(H.shape[0]), (X.T @ y) / m)  # ~OLS optimum
evals, Q = np.linalg.eigh(H)
print('Hessian shape', H.shape, '| eigenvalue range',
      f'{evals.min():.3f} .. {evals.max():.3f}')

In [ ]:
# Rescaling factor lambda/(lambda+alpha) per eigen-direction, for several alphas.
plt.figure(figsize=(8,5))
order = np.argsort(evals)                       # flat directions first
for alpha in [0.05, 0.2, 1.0]:
    factors = evals[order] / (evals[order] + alpha)
    plt.plot(factors, marker='.', label=f'α={alpha}')
plt.axhline(0.5, color='gray', ls=':', lw=1)
plt.text(1, 0.52, 'half-life: α = λ', color='gray', fontsize=8)
plt.xlabel('eigen-direction (flat → steep)'); plt.ylabel('surviving fraction  λ/(λ+α)')
plt.title('Station 3 — L² keeps steep directions, decays flat ones')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# How much does the whole weight vector shrink as alpha grows?
print('alpha |   ||w*||   ->   ||w~||   | shrink')
for alpha in [0.0, 0.05, 0.2, 1.0, 5.0]:
    w_t = np.linalg.solve(H + alpha*np.eye(H.shape[0]), H @ w_star)
    shrink = 1 - np.linalg.norm(w_t)/np.linalg.norm(w_star)
    print(f'{alpha:5.2f} |  {np.linalg.norm(w_star):6.3f}   ->  {np.linalg.norm(w_t):6.3f}  | {shrink*100:5.1f}%')
print('\nThis is ridge regression: w = (X^T X + alpha I)^{-1} X^T y.')
print('Adding alpha*I on the diagonal is what tames correlated/low-variance features.')

## Station 4 — L¹ regularization & sparsity

Swap the squared norm for the sum of absolute values:

$$\Omega(\theta)=\lVert w\rVert_1=\sum_i|w_i|,\qquad \nabla_w\tilde J = \nabla_w J + \alpha\,\mathrm{sign}(w).$$

The penalty gradient `α·sign(w)` is a **constant** push toward zero — it does *not* shrink as `w` shrinks, so it finishes the job and pins weights at **exactly 0**. That's sparsity (feature selection), a.k.a. the **LASSO**. Under a diagonal-Hessian approximation the solution is **soft-thresholding**:

$$\tilde w_i = \mathrm{sign}(w^*_i)\,\max\!\big(|w^*_i|-\alpha/H_{ii},\,0\big).$$

First the shape of the two shrinkage rules, then the real thing on MNIST.


In [ ]:
# Soft-threshold (L1) vs proportional shrink (L2), the shrinkage bench.
w = np.linspace(-3, 3, 400)
alpha, Hh = 0.6, 1.0
l1 = np.sign(w) * np.maximum(np.abs(w) - alpha/Hh, 0.0)   # dead zone then parallel line
l2 = w * (Hh / (Hh + alpha))                              # line through origin, slope<1

plt.figure(figsize=(6.5,6))
plt.plot(w, w,  color='gray', ls='--', label='no reg  (w̃ = w*)')
plt.plot(w, l1, color='crimson', lw=2, label='L¹ soft-threshold')
plt.plot(w, l2, color='purple',  lw=2, label='L² proportional')
plt.axvspan(-alpha/Hh, alpha/Hh, color='crimson', alpha=0.08)
plt.text(0, -2.5, 'L¹ dead-zone ±α/H\n(exact zeros)', ha='center', color='crimson', fontsize=8)
plt.xlabel('unregularized weight w*'); plt.ylabel('output weight w̃')
plt.legend(); plt.title('Station 4 — L¹ pins small weights to 0; L² only trims'); plt.grid(alpha=0.3)
plt.show()

### L¹ on a real MNIST classifier (with true zeros)

A plain L¹ *penalty* trained by SGD rarely produces exact zeros because floating-point gradients wiggle weights around 0. To get genuine sparsity we take the **proximal / ISTA** step the lesson shows: an ordinary gradient step on the data term, then a soft-threshold that clamps sub-threshold weights to exactly 0.


In [ ]:
def train_linear_l1(l1_lambda=1e-3, lr=0.1, epochs=40):
    lin = nn.Linear(28*28, 10).to(device)
    opt = torch.optim.SGD(lin.parameters(), lr=lr)   # data-term gradient only
    for ep in range(epochs):
        lin.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            F.cross_entropy(lin(xb.view(xb.size(0), -1)), yb).backward()
            opt.step()
            with torch.no_grad():                     # proximal soft-threshold on WEIGHTS
                t = lr * l1_lambda
                w = lin.weight
                lin.weight.copy_(torch.sign(w) * torch.clamp(w.abs() - t, min=0.0))
    return lin

lin_l1 = train_linear_l1(l1_lambda=1e-2)
# For contrast: an L2 linear model (weight decay, no thresholding).
torch.manual_seed(0)
lin_l2 = nn.Linear(28*28, 10).to(device)
opt = torch.optim.SGD(lin_l2.parameters(), lr=0.1, weight_decay=1e-2)
for ep in range(40):
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); F.cross_entropy(lin_l2(xb.view(xb.size(0),-1)), yb).backward(); opt.step()

def frac_zero(layer, tol=1e-8):
    w = layer.weight.detach()
    return (w.abs() < tol).float().mean().item()

print(f'L¹ model: {frac_zero(lin_l1)*100:5.1f}% of weights are EXACTLY zero | '
      f'val acc {evaluate(lin_l1, val_loader)[1]:.3f}')
print(f'L² model: {frac_zero(lin_l2)*100:5.1f}% of weights are exactly zero | '
      f'val acc {evaluate(lin_l2, val_loader)[1]:.3f}')

In [ ]:
# Which pixels survive? Visualize the per-class weight images (L1 vs L2).
fig, axes = plt.subplots(2, 5, figsize=(11,4.6))
for d in range(5):
    axes[0,d].imshow(lin_l1.weight[d].detach().cpu().reshape(28,28), cmap='RdBu')
    axes[0,d].set_title(f'L¹ · "{d}"', fontsize=9)
    axes[1,d].imshow(lin_l2.weight[d].detach().cpu().reshape(28,28), cmap='RdBu')
    axes[1,d].set_title(f'L² · "{d}"', fontsize=9)
for a in axes.ravel(): a.axis('off')
plt.suptitle('L¹ zeroes out whole regions (white) — it SELECTS pixels; L² keeps everything faint')
plt.tight_layout(); plt.show()

## Station 5 — L¹ vs L² and the priors behind them

Both penalties fall out of the same Bayesian idea: put a **prior** on the weights that says "small is more probable," then do **MAP** estimation. The log-prior *is* the negative penalty `−αΩ(w)`:

- **Gaussian prior** `N(0, 1/α)` → L² (smooth through 0).
- **Laplace prior** → L¹ (a sharp **cusp at 0** and heavier tails).

That cusp is the probabilistic origin of the dead-zone: MAP is happy to park a weight exactly at the peak. We plot the two prior shapes, then histogram the weights our two MNIST models actually learned.


In [ ]:
x = np.linspace(-3, 3, 400)
b = 0.5
gauss = np.exp(-x**2 / (2*b**2))
lap   = np.exp(-np.abs(x) / b)

fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
ax[0].plot(x, gauss/gauss.max(), color='purple',  lw=2, label='Gaussian prior → L²')
ax[0].plot(x, lap/lap.max(),     color='crimson', lw=2, label='Laplace prior → L¹')
ax[0].set_title('The priors: Laplace has a cusp at 0'); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].hist(lin_l2.weight.detach().cpu().numpy().ravel(), bins=120, alpha=0.6,
           color='purple', label='L² weights')
ax[1].hist(lin_l1.weight.detach().cpu().numpy().ravel(), bins=120, alpha=0.6,
           color='crimson', label='L¹ weights (spike at 0)')
ax[1].set_yscale('log'); ax[1].set_title('Learned MNIST weights: L¹ piles mass at 0')
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

| Property | L² · ridge / weight decay | L¹ · LASSO |
|---|---|---|
| Penalty gradient | `α w` (shrinks with `w`) | `α sign(w)` (constant) |
| Effect | proportional shrink `λ/(λ+α)` | soft threshold — many exact 0 |
| Sparsity | no | yes (feature selection) |
| Bayesian prior | Gaussian | Laplace |
| Constraint ball | circle / sphere (smooth) | diamond (cornered) |

**Elastic Net** mixes both — sparsity *and* stability — and is the go-to when features come in correlated groups (`l1_ratio ≈ 0.5`, tuned on validation).


## Station 6 — Penalties as constrained optimization

There's a second way to read every norm penalty: as a hard **constraint** that the weights stay inside a region `Ω(w) < k`. The generalized Lagrangian links them:

$$\mathcal L(\theta,\alpha)=J(\theta)+\alpha\big(\Omega(\theta)-k\big).$$

Fixing the optimal `α*` turns this back into the penalized objective from station 1 — penalty and constraint are two faces of one problem. **α and k move oppositely:** a larger `α` implies a smaller region `k`. You rarely know the exact `k` a given `α` implies, but you always know the direction. We can *measure* the implied `k` on our 2-D bowl: it's just `‖w̃(α)‖`.


In [ ]:
alphas = np.logspace(-2, 2, 60)
k_of_alpha = [np.linalg.norm(w_tilde_2d(a)) for a in alphas]

plt.figure(figsize=(7.5,5))
plt.semilogx(alphas, k_of_alpha, color='crimson', lw=2)
plt.xlabel('regularization strength α  (penalty view)')
plt.ylabel('implied region radius k = ‖w̃‖  (constraint view)')
plt.title('Station 6 — bigger α ⇔ smaller constraint region k')
plt.grid(alpha=0.3, which='both'); plt.show()
print('Same solution, two knobs: add αΩ to the loss, OR optimize freely and clip w into a ball of radius k.')
print('In practice you cross-validate α on a log grid (1e-4, 1e-3, ... 1e1) and keep the best.')

## Station 7 — Explicit constraints & reprojection (max-norm)

Take the constraint literally: after each gradient step, **project** the weights back onto the region. This *projected gradient descent* often behaves better than the equivalent penalty:

- **No dead zones** near the origin where non-convex optimization stalls with all-tiny weights.
- **Stability** — it caps the norm and prevents the runaway feedback loop (big weights → big gradients → bigger weights).
- **Max-norm** constrains the norm of **each column** (each hidden unit) separately, so no unit shouts louder than the rest. It was famously paired with dropout to allow aggressive learning rates.

Our `train(max_norm=c)` already reprojects each row (unit) to norm ≤ `c` after every step. Let's confirm the cap bites and compare training.


In [ ]:
torch.manual_seed(0)
m_free = MLP(hidden=256)
h_free = train(m_free, epochs=60, lr=0.05, verbose_every=0)          # no constraint

torch.manual_seed(0)
m_mn = MLP(hidden=256)
h_mn = train(m_mn, epochs=60, lr=0.05, max_norm=3.0, verbose_every=0)  # max-norm = 3

cn_free = m_free.fc1.weight.norm(dim=1).detach().cpu().numpy()
cn_mn   = m_mn.fc1.weight.norm(dim=1).detach().cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(13,4.6))
ax[0].hist(cn_free, bins=40, alpha=0.6, color='gray',    label=f'free (max {cn_free.max():.1f})')
ax[0].hist(cn_mn,   bins=40, alpha=0.6, color='crimson', label=f'max-norm=3 (max {cn_mn.max():.1f})')
ax[0].axvline(3.0, color='crimson', ls='--'); ax[0].set_title('Per-unit incoming-weight norms')
ax[0].set_xlabel('column norm'); ax[0].legend()
ax[1].plot(h_free['val_acc'], label='free'); ax[1].plot(h_mn['val_acc'], label='max-norm=3')
ax[1].set_title('Validation accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()
print(f'Max column norm — free: {cn_free.max():.2f} | constrained: {cn_mn.max():.2f} (capped at 3.0)')

Every unit under the cap is left alone; any unit that grows past it is rescaled straight back to the boundary. Its close cousin, **gradient clipping** (clip norm to ~1.0), is standard for RNNs and Transformers to survive gradient spikes.


## Station 8 — Regularization & under-constrained problems

Some models have **no unique answer** without regularization. When the data has flat directions (no variance, or fewer examples than parameters), the matrix you must invert is **singular** and the optimum runs off to infinity. A norm penalty makes the problem well-posed.

We show three faces of this: (a) `XᵀX + αI` restores invertibility; (b) separable logistic regression sends `‖w‖ → ∞` until weight decay halts it; (c) the pseudoinverse is the `α → 0⁺` limit of ridge.


### (a) The singular matrix, fixed by αI

With more features than samples (`p ≫ n`), `XᵀX` is rank-deficient — its smallest eigenvalue is 0, so it can't be inverted and the condition number is infinite. Adding `α` to every diagonal entry lifts every eigenvalue by `α`, so the smallest can no longer be zero.


In [ ]:
# 30 samples, 196 features (14x14)  ->  p >> n  ->  X^T X is singular.
X_uc, _ = get_xy(train_full, list(range(10)), n=30, pool=2)
G = X_uc.T @ X_uc
print(f'X^T X is {G.shape[0]}x{G.shape[0]} but rank {np.linalg.matrix_rank(G)} (<= 30 samples) -> singular\n')
print('alpha  |  min eigenvalue  |  invertible? |  condition number')
for a in [0.0, 1e-3, 1e-1, 1.0]:
    ev = np.linalg.eigvalsh(G + a*np.eye(G.shape[0]))
    lo = ev.min()
    kappa = ev.max()/lo if lo > 1e-9 else np.inf
    print(f'{a:5.3f}  |  {lo:13.2e}  |  {"yes" if lo>1e-9 else "NO ":>10}   |  {kappa:.2e}')

### (b) Separable data → runaway weights

Logistic regression on **perfectly separable** classes has no finite optimum: if `w` separates the data then `2w` gives higher likelihood, and so on forever — `‖w‖ → ∞`. Any weight decay halts the runaway at a finite point where the likelihood gain finally equals the decay cost. MNIST's 0-vs-1, downsampled, is linearly separable — perfect for the demo.


In [ ]:
Xs, ys = get_xy(train_full, [0, 1], n=200, pool=4)
Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8)
Xt = torch.tensor(Xs, dtype=torch.float32)
yt = torch.tensor((ys == 1).astype(np.float32))

def run_logreg(weight_decay, steps=1500, lr=0.5):
    w = torch.zeros(Xt.shape[1], requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.SGD([w, b], lr=lr, weight_decay=weight_decay)
    norms = []
    for _ in range(steps):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(Xt @ w + b, yt)
        loss.backward(); opt.step()
        norms.append(w.detach().norm().item())
    return norms

plt.figure(figsize=(8,5))
for wd, style in [(0.0, '--'), (1e-3, '-'), (1e-2, '-')]:
    plt.plot(run_logreg(wd), style, label=f'weight_decay={wd}')
plt.xlabel('gradient step'); plt.ylabel('‖w‖')
plt.title('Station 8 — separable data: ‖w‖ runs away at α=0, plateaus with any decay')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('Dashed (α=0) keeps climbing — the optimum is at infinity.')
print('This is why sklearn LogisticRegression is L2-regularized by default.')

### (c) The pseudoinverse as a regularization limit

The Moore–Penrose pseudoinverse — the tool that gives a stable, minimum-norm answer to under-determined systems — is exactly the vanishing-regularization limit of ridge:

$$X^+ = \lim_{\alpha\to 0^+}(X^\top X+\alpha I)^{-1}X^\top.$$

"Regularize, then let the leash go slack." What remains is the minimum-norm solution regularization was quietly selecting all along.


In [ ]:
Xp, yp = get_xy(train_full, [0, 1], n=30, pool=2)   # p >> n again
yp = (yp == 1).astype(np.float64)
w_pinv = np.linalg.pinv(Xp) @ yp                     # min-norm solution

print('alpha    | ||w_ridge - w_pinv||  (ridge -> pseudoinverse as alpha -> 0)')
for a in [1.0, 1e-1, 1e-2, 1e-4, 1e-8]:
    w_ridge = np.linalg.solve(Xp.T @ Xp + a*np.eye(Xp.shape[1]), Xp.T @ yp)
    print(f'{a:8.0e} | {np.linalg.norm(w_ridge - w_pinv):.3e}')
print('\nAs alpha shrinks, the ridge solution converges to the pseudoinverse (min-norm) answer.')

## Key takeaways — what you carry off the landscape

1. **Penalty, not punishment.** `J̃ = J + αΩ` reduces *generalization* error, not training error. `α` is a validation-tuned dial; we penalize weights, not biases.
2. **L² is a per-step shrink.** Weight decay multiplies `w` by `(1 − εα)` before each step; its effective strength is entangled with the learning rate — hence AdamW.
3. **Curvature decides survival.** `w̃ = (H + αI)⁻¹Hw*`; each direction scales by `λ/(λ+α)` — steep kept, flat decayed.
4. **L¹ is soft-thresholding.** A constant push `α·sign(w)` pins small weights at exactly 0 → sparsity, a.k.a. LASSO.
5. **Priors behind the penalties.** L² = MAP with a Gaussian prior; L¹ = MAP with a Laplace prior. The Laplace cusp at 0 is where sparsity comes from.
6. **Penalty ⇄ constraint.** Via `J + α(Ω−k)`, strength `α` equals some region size `k`; more `α` ⇒ smaller region.
7. **Explicit constraints are steadier.** Reproject after each step: no dead zones, no runaway divergence, and per-column max-norm.
8. **Regularization makes problems well-posed.** `XᵀX + αI` is always invertible; separable logistic regression gets a finite solution; the pseudoinverse is the `α → 0` limit of ridge.

### Try it yourself
- Re-run station 0's overfit model with `p_drop=0.5` (dropout) — another regularizer, and see the gap shrink.
- Push `N_TRAIN` up to 10,000 and watch how much *less* regularization you need — data is the ultimate regularizer.
- Sweep the L¹ `l1_lambda` and plot sparsity (% exact zeros) vs validation accuracy — find the sweet spot.

*Reference: I. Goodfellow, Y. Bengio, A. Courville, "Deep Learning" (MIT Press), Chapter 7, §7.1–7.3.*
